In [1]:
import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
#BLOQUE 2 (Ruta del JSON en directorio en el que estamos)
NOMBRE_PROGRAMA = "Extrae_Institucion.ipynb"


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root

        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(

    ruta_programa,

    "JSONs"

)

os.makedirs(

    CARPETA_JSON,

    exist_ok=True

)

In [3]:
# ==========================================================
# BLOQUE 3. CONFIGURACIÓN DE LA FUENTE
#
# Fuente: La institución - UPV
#
# Este bloque define:
# - URL de la página raíz
# - nombre y ruta del JSON
# - secciones semánticas de la página
# - cabeceras HTTP
# - funciones auxiliares de normalización
# - estructura base que tendrá el JSON
#
# CARPETA_JSON y ruta_programa se calculan previamente.
# ==========================================================

import os
import json
import re
import unicodedata
from urllib.parse import urljoin, urlparse


# ==========================================================
# 1. CONFIGURACIÓN GENERAL
# ==========================================================

URL_RAIZ = (
    "https://www.upv.es/organizacion/"
    "la-institucion/index-es.html"
)

NOMBRE_JSON = "institucion.json"

RUTA_JSON = os.path.join(
    CARPETA_JSON,
    NOMBRE_JSON
)


# ==========================================================
# 2. SECCIONES SEMÁNTICAS
#
# Estas categorías representan los cinco recursos
# principales que se quieren extraer de la página raíz.
# ==========================================================

SECCIONES_VALIDAS = {
    "Órganos de gobierno",
    "La UPV al detalle",
    "Publicaciones oficiales",
    "Estrategia UPV_SIRVE",
    "Sindicatura"
}


# ==========================================================
# 3. CABECERAS HTTP
# ==========================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,image/avif,image/webp,"
        "*/*;q=0.8"
    ),
    "Accept-Language": (
        "es-ES,es;q=0.9,en;q=0.8"
    ),
    "Connection": "keep-alive"
}


# ==========================================================
# 4. FUNCIONES AUXILIARES
# ==========================================================

def limpiar_texto(texto):
    """
    Limpia y normaliza un fragmento de texto extraído
    del HTML.

    Elimina espacios, saltos de línea y tabulaciones
    generados por la estructura HTML sin alterar el
    contenido textual.
    """

    if texto is None:
        return ""

    texto = str(texto)

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


# ----------------------------------------------------------
# Normalización de nombres para identificadores
# ----------------------------------------------------------

def normalizar_identificador(texto):
    """
    Convierte un texto en un identificador estable.

    Ejemplo:
        'Órganos de gobierno'
        -> 'organos_de_gobierno'
    """

    texto = limpiar_texto(texto)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")


# ----------------------------------------------------------
# Convertir URL relativa en absoluta
# ----------------------------------------------------------

def normalizar_url(
    url,
    url_base=URL_RAIZ
):
    """
    Convierte una URL relativa en absoluta utilizando
    la URL de la página desde la que se ha obtenido.
    """

    if not url:
        return ""

    url = url.strip()

    return urljoin(
        url_base,
        url
    )


# ----------------------------------------------------------
# Comprobar URL HTTP/HTTPS
# ----------------------------------------------------------

def es_url_valida(url):
    """
    Comprueba si una URL corresponde a HTTP o HTTPS.
    """

    if not url:
        return False

    try:

        analisis = urlparse(url)

        return analisis.scheme in {
            "http",
            "https"
        }

    except Exception:

        return False


# ----------------------------------------------------------
# Deduplicar elementos conservando el orden
# ----------------------------------------------------------

def deduplicar_lista(
    elementos,
    clave="url"
):
    """
    Elimina elementos duplicados de una lista de
    diccionarios.

    Se conserva la primera aparición.
    """

    resultado = []
    vistos = set()

    for elemento in elementos:

        if not isinstance(
            elemento,
            dict
        ):
            continue

        valor = elemento.get(
            clave,
            ""
        )

        if valor in vistos:
            continue

        vistos.add(valor)

        resultado.append(
            elemento
        )

    return resultado


# ----------------------------------------------------------
# Crear estructura vacía de una sección
# ----------------------------------------------------------

def crear_seccion(
    titulo,
    tipo="seccion"
):
    """
    Crea la estructura base de una sección semántica.
    """

    return {

        "id": normalizar_identificador(
            titulo
        ),

        "titulo": limpiar_texto(
            titulo
        ),

        "tipo": tipo,

        "descripcion": "",

        "elementos": []
    }


# ----------------------------------------------------------
# Crear elemento documental
# ----------------------------------------------------------

def crear_elemento(
    titulo="",
    descripcion="",
    url="",
    tipo="recurso"
):
    """
    Crea una estructura homogénea para los recursos
    extraídos de las distintas secciones.
    """

    return {

        "tipo": tipo,

        "titulo": limpiar_texto(
            titulo
        ),

        "descripcion": limpiar_texto(
            descripcion
        ),

        "url": normalizar_url(
            url
        )
    }


# ==========================================================
# 5. ESTRUCTURA BASE DEL JSON
#
# El JSON representa la organización semántica de la
# página de La institución, no la estructura técnica
# concreta del HTML.
#
# La página principal será el documento padre.
# Las cinco secciones serán recursos que posteriormente
# se visitarán y extraerán individualmente.
# ==========================================================

def crear_json_base():
    """
    Construye la estructura inicial del JSON de
    La institución.
    """

    return {

        # --------------------------------------------------
        # Información general de la página
        # --------------------------------------------------

        "titulo": "La institución",

        "url": URL_RAIZ,

        "tipo": "padre",

        # --------------------------------------------------
        # Secciones / recursos principales
        # --------------------------------------------------

        "secciones": [

            crear_seccion(
                "Órganos de gobierno",
                "organos_gobierno"
            ),

            crear_seccion(
                "La UPV al detalle",
                "upv_al_detalle"
            ),

            crear_seccion(
                "Publicaciones oficiales",
                "publicaciones_oficiales"
            ),

            crear_seccion(
                "Estrategia UPV_SIRVE",
                "estrategia_upv_sirve"
            ),

            crear_seccion(
                "Sindicatura",
                "sindicatura"
            )
        ]
    }


# ==========================================================
# 6. CREAR JSON INICIAL
# ==========================================================

json_institucion = crear_json_base()


# ==========================================================
# 7. RUTAS DE MARKDOWN
# ==========================================================

# Carpeta principal de la fuente

CARPETA_INSTITUCION = os.path.join(
    ruta_programa,
    "INSTITUCION"
)


# ----------------------------------------------------------
# Carpetas correspondientes a las cinco secciones
# ----------------------------------------------------------

CARPETA_ORGANOS_GOBIERNO = os.path.join(
    CARPETA_INSTITUCION,
    "organos_gobierno"
)

CARPETA_UPV_AL_DETALLE = os.path.join(
    CARPETA_INSTITUCION,
    "upv_al_detalle"
)

CARPETA_PUBLICACIONES_OFICIALES = os.path.join(
    CARPETA_INSTITUCION,
    "publicaciones_oficiales"
)

CARPETA_ESTRATEGIA_UPV_SIRVE = os.path.join(
    CARPETA_INSTITUCION,
    "estrategia_upv_sirve"
)

CARPETA_SINDICATURA = os.path.join(
    CARPETA_INSTITUCION,
    "sindicatura"
)


# ----------------------------------------------------------
# Crear directorios
# ----------------------------------------------------------

CARPETAS_SECCIONES = [
    CARPETA_ORGANOS_GOBIERNO,
    CARPETA_UPV_AL_DETALLE,
    CARPETA_PUBLICACIONES_OFICIALES,
    CARPETA_ESTRATEGIA_UPV_SIRVE,
    CARPETA_SINDICATURA
]

for carpeta in CARPETAS_SECCIONES:

    os.makedirs(
        carpeta,
        exist_ok=True
    )


# ----------------------------------------------------------
# Markdown de la página padre
# ----------------------------------------------------------

RUTA_MARKDOWN_PADRE = os.path.join(
    CARPETA_INSTITUCION,
    "institucion.md"
)


# ==========================================================
# 8. COMPROBACIÓN DE RUTAS
# ==========================================================

print()
print("=" * 70)
print("ESTRUCTURA DE MARKDOWN - LA INSTITUCIÓN")
print("=" * 70)

print()
print("Carpeta principal:")
print(CARPETA_INSTITUCION)

print()
print("Markdown página padre:")
print(RUTA_MARKDOWN_PADRE)

print()
print("Carpetas de las secciones:")

print(
    "  - Órganos de gobierno:",
    CARPETA_ORGANOS_GOBIERNO
)

print(
    "  - La UPV al detalle:",
    CARPETA_UPV_AL_DETALLE
)

print(
    "  - Publicaciones oficiales:",
    CARPETA_PUBLICACIONES_OFICIALES
)

print(
    "  - Estrategia UPV_SIRVE:",
    CARPETA_ESTRATEGIA_UPV_SIRVE
)

print(
    "  - Sindicatura:",
    CARPETA_SINDICATURA
)

print()
print("OK: estructura de directorios preparada.")

print()
print("=" * 70)


ESTRUCTURA DE MARKDOWN - LA INSTITUCIÓN

Carpeta principal:
/content/drive/MyDrive/TFG Teleco/INSTITUCION

Markdown página padre:
/content/drive/MyDrive/TFG Teleco/INSTITUCION/institucion.md

Carpetas de las secciones:
  - Órganos de gobierno: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno
  - La UPV al detalle: /content/drive/MyDrive/TFG Teleco/INSTITUCION/upv_al_detalle
  - Publicaciones oficiales: /content/drive/MyDrive/TFG Teleco/INSTITUCION/publicaciones_oficiales
  - Estrategia UPV_SIRVE: /content/drive/MyDrive/TFG Teleco/INSTITUCION/estrategia_upv_sirve
  - Sindicatura: /content/drive/MyDrive/TFG Teleco/INSTITUCION/sindicatura

OK: estructura de directorios preparada.



In [4]:
# ==========================================================
# BLOQUE 4. EXTRACCIÓN DE LA ESTRUCTURA DE LA INSTITUCIÓN
#
# Este bloque:
#
# - descarga y analiza la página principal
# - localiza las cinco secciones documentales
# - identifica el contenedor específico de cada sección
# - extrae únicamente sus recursos y URLs
# - conserva la estructura semántica
# - ignora navegación, footer y contenido no relevante
#
# Las URLs de los recursos se procesarán posteriormente
# para generar su Markdown individual.
# ==========================================================

import requests
from bs4 import BeautifulSoup


# ==========================================================
# 1. DESCARGAR LA PÁGINA
# ==========================================================

respuesta = requests.get(
    URL_RAIZ,
    headers=HEADERS,
    timeout=30
)

respuesta.raise_for_status()

soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)


# ==========================================================
# 2. LOCALIZAR EL CONTENEDOR PRINCIPAL
# ==========================================================

contenedor_principal = soup.find(
    id="smooth-wrapper"
)

if contenedor_principal is None:

    raise Exception(
        "No se ha encontrado el contenedor principal #smooth-wrapper."
    )


# ==========================================================
# 3. DEFINICIÓN DE LAS CINCO SECCIONES
# ==========================================================

CONFIG_SECCIONES = {

    "Organos de gobierno": {
        "titulo": "Órganos de gobierno",
        "tipo": "organos_gobierno"
    },

    "Publicaciones oficiales": {
        "titulo": "Publicaciones oficiales",
        "tipo": "publicaciones_oficiales"
    },

    "La UPV al detalle": {
        "titulo": "La UPV al detalle",
        "tipo": "upv_al_detalle"
    },

    "Estrategia UPV_SIRVE": {
        "titulo": "Estrategia UPV_SIRVE",
        "tipo": "estrategia_upv_sirve"
    },

    "Sindicatura": {
        "titulo": "Sindicatura",
        "tipo": "sindicatura"
    }
}


# ==========================================================
# 4. FUNCIÓN PARA COMPARAR TÍTULOS
# ==========================================================

def titulo_normalizado(texto):

    texto = limpiar_texto(texto)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    return texto.lower().strip()


# ==========================================================
# 5. LOCALIZAR UNA SECCIÓN
#
# En esta página la estructura HTML es diferente según
# la sección:
#
# - Órganos de gobierno:
#       section
#         └── div.main-card
#
# - Resto de secciones:
#       section
#         └── div.secondary-card
#
# Por ello buscamos primero el encabezado y, a partir de
# él, ascendemos hasta la tarjeta correspondiente.
# ==========================================================

def localizar_seccion(titulo_objetivo):

    objetivo = titulo_normalizado(
        titulo_objetivo
    )

    # ------------------------------------------------------
    # Buscar únicamente dentro del contenido principal
    # ------------------------------------------------------

    for encabezado in contenedor_principal.find_all(
        ["h2", "h3"]
    ):

        texto = limpiar_texto(
            encabezado.get_text(
                " ",
                strip=True
            )
        )

        if titulo_normalizado(texto) != objetivo:
            continue

        # --------------------------------------------------
        # Caso 1:
        # Órganos de gobierno
        #
        # Su estructura es:
        #
        # <section>
        #   ...
        #   <div class="main-card">
        #       <h2>Órganos de gobierno</h2>
        #       ...
        #   </div>
        # </section>
        #
        # Buscamos primero la tarjeta.
        # --------------------------------------------------

        tarjeta_principal = encabezado.find_parent(
            class_="main-card"
        )

        if tarjeta_principal is not None:
            return tarjeta_principal

        # --------------------------------------------------
        # Caso 2:
        # Resto de secciones
        #
        # Su estructura es:
        #
        # <section>
        #   ...
        #   <div class="secondary-card">
        #       <h3>...</h3>
        #       ...
        #   </div>
        # </section>
        # --------------------------------------------------

        tarjeta_secundaria = encabezado.find_parent(
            class_="secondary-card"
        )

        if tarjeta_secundaria is not None:
            return tarjeta_secundaria

        # --------------------------------------------------
        # Caso de respaldo:
        # si la estructura cambia, buscamos la section.
        # --------------------------------------------------

        section = encabezado.find_parent(
            "section"
        )

        if section is not None:
            return section

    return None


# ==========================================================
# 6. EXTRAER RECURSOS DE UNA SECCIÓN
# ==========================================================

def extraer_recursos_seccion(
    elemento,
    titulo_seccion
):

    recursos = []

    # ------------------------------------------------------
    # Los enlaces se buscan EXCLUSIVAMENTE dentro del
    # elemento correspondiente a esta tarjeta.
    #
    # Esto evita que una sección absorba los enlaces de
    # las tarjetas posteriores.
    # ------------------------------------------------------

    for enlace in elemento.find_all("a"):

        titulo = limpiar_texto(
            enlace.get_text(
                " ",
                strip=True
            )
        )

        url = enlace.get(
            "href",
            ""
        )

        if not titulo or not url:
            continue

        # --------------------------------------------------
        # Normalizar URL
        # --------------------------------------------------

        url = normalizar_url(
            url,
            URL_RAIZ
        )

        if not es_url_valida(url):
            continue

        # --------------------------------------------------
        # Eliminar enlaces genéricos de interfaz
        # --------------------------------------------------

        if titulo_normalizado(titulo) in {
            "mas informacion",
            "mas info",
            "ver mas"
        }:
            continue

        # --------------------------------------------------
        # Crear recurso
        # --------------------------------------------------

        recursos.append(
            crear_elemento(
                titulo=titulo,
                descripcion="",
                url=url,
                tipo="recurso"
            )
        )

    # ------------------------------------------------------
    # Eliminar duplicados manteniendo el orden
    # ------------------------------------------------------

    recursos = deduplicar_lista(
        recursos,
        clave="url"
    )

    return recursos


# ==========================================================
# 7. EXTRAER LAS CINCO SECCIONES
# ==========================================================

secciones_extraidas = []

for clave, configuracion in CONFIG_SECCIONES.items():

    titulo = configuracion["titulo"]

    # ------------------------------------------------------
    # Localizar la tarjeta concreta
    # ------------------------------------------------------

    elemento = localizar_seccion(
        titulo
    )

    if elemento is None:

        print(
            f"AVISO: no se ha encontrado: {titulo}"
        )

        continue

    # ------------------------------------------------------
    # Extraer únicamente los enlaces contenidos en esa
    # tarjeta
    # ------------------------------------------------------

    recursos = extraer_recursos_seccion(
        elemento,
        titulo
    )

    # ------------------------------------------------------
    # Crear la sección
    # ------------------------------------------------------

    seccion = crear_seccion(
        titulo,
        configuracion["tipo"]
    )

    seccion["elementos"] = recursos

    secciones_extraidas.append(
        seccion
    )


# ==========================================================
# 8. CONSTRUIR EL JSON
# ==========================================================

json_institucion = {

    "titulo": "La institución",

    "url": URL_RAIZ,

    "tipo": "padre",

    "secciones": secciones_extraidas
}


# ==========================================================
# 9. MOSTRAR RESULTADO
# ==========================================================

print()
print("=" * 70)
print("ESTRUCTURA EXTRAÍDA - LA INSTITUCIÓN")
print("=" * 70)

print()

print(
    "Título:",
    json_institucion["titulo"]
)

print(
    "URL:",
    json_institucion["url"]
)

print()

for seccion in json_institucion["secciones"]:

    print(
        f"[{seccion['titulo']}]"
    )

    for elemento in seccion["elementos"]:

        print(
            f"  - {elemento['titulo']}"
        )

        print(
            f"    {elemento['url']}"
        )

    print()


print(
    "Número de secciones:",
    len(
        json_institucion["secciones"]
    )
)

print(
    "Número total de recursos:",
    sum(
        len(seccion["elementos"])
        for seccion in json_institucion["secciones"]
    )
)

print()

print("=" * 70)

import json

print()
print("=" * 70)
print("JSON - LA INSTITUCIÓN")
print("=" * 70)
print()

print(
    json.dumps(
        json_institucion,
        ensure_ascii=False,
        indent=2
    )
)

print()
print("=" * 70)


ESTRUCTURA EXTRAÍDA - LA INSTITUCIÓN

Título: La institución
URL: https://www.upv.es/organizacion/la-institucion/index-es.html

[Órganos de gobierno]
  - Unipersonales y colegiados
    https://www.upv.es/organizacion/la-institucion/organos-gobierno-upv-es.html
  - Rector
    https://rector.upv.es/es
  - Equipo rectoral
    https://www.upv.es/organizacion/la-institucion/equipo-rectoral-es.html
  - Vicerrectorados y sus áreas
    https://www.upv.es/organizacion/la-institucion/vicerrectorados-y-areas-es.html
  - Secretaria General
    https://www.upv.es/entidades/SG/index-es.html
  - Gerencia
    https://www.upv.es/entidades/ger/
  - Consejo Social
    https://www.upv.es/entidades/CS/index-es.html

[Publicaciones oficiales]
  - Boletín oficial de la UPV
    https://www.upv.es/entidades/SG/infoweb/sg/info/513084normalc.html
  - Estatutos
    https://www.upv.es/organizacion/la-institucion/estatutos-upv-es.html
  - Memoria del curso académico
    https://www.upv.es/organizacion/la-instituci

In [5]:
# ==========================================================
# BLOQUE 5. GUARDAR EL JSON
# ==========================================================

RUTA_JSON = os.path.join(
    CARPETA_JSON,
    "institucion.json"
)

with open(
    RUTA_JSON,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        json_institucion,
        archivo,
        ensure_ascii=False,
        indent=2
    )


# ==========================================================
# COMPROBACIÓN
# ==========================================================

print()
print("=" * 70)
print("JSON GUARDADO CORRECTAMENTE")
print("=" * 70)

print()
print("Archivo JSON:")
print(RUTA_JSON)

print()
print(
    "Número de secciones:",
    len(json_institucion["secciones"])
)

print(
    "Número total de recursos:",
    sum(
        len(seccion["elementos"])
        for seccion in json_institucion["secciones"]
    )
)

print()
print("=" * 70)


JSON GUARDADO CORRECTAMENTE

Archivo JSON:
/content/drive/MyDrive/TFG Teleco/JSONs/institucion.json

Número de secciones: 5
Número total de recursos: 26



In [8]:
# ==========================================================
# BLOQUE 6. GENERACIÓN DE MARKDOWN (REESCRITO)
#
# Cambios respecto a la versión anterior:
#
# 1. Se corrige el cruce sección->carpeta usando "id" del
#    JSON en vez de "tipo" (ahí estaba el bug de
#    'organos_gobierno' que se saltaba entero).
# 2. La extracción de contenido ya no se limita a
#    h1/h2/h3/h4/p/li: recorre cualquier elemento "hoja"
#    (sin bloques anidados dentro), incluidos <a> sueltos,
#    para no perder nombres, teléfonos, emails y enlaces
#    "Más información". Los enlaces se convierten en
#    Markdown [texto](url) en vez de perder el href.
# 3. Se añaden metadatos YAML homogéneos en todos los .md,
#    con el formato exacto solicitado.
# 4. Si una página tiene poco contenido, se visitan hasta
#    MAX_ENLACES_HIJOS enlaces internos encontrados en el
#    contenedor principal y se añade un resumen de cada uno.
# 5. Si la URL no devuelve HTML (PDF, vídeo...), se genera
#    igualmente el .md con metadatos y un aviso, sin error.
# ==========================================================

import os
import re
import time
import unicodedata
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


# ==========================================================
# 0. CONFIGURACIÓN DE METADATOS Y UMBRALES
# ==========================================================

FUENTE = "UPV"
CATEGORIA = "institucion"
NIVEL = "institucional"
PADRE_SLUG = "la_institucion"

UMBRAL_PALABRAS_POCO_CONTENIDO = 60
MAX_ENLACES_HIJOS = 5
MAX_CARACTERES_FRAGMENTO_HIJO = 800

TAGS_TITULO = {"h1", "h2", "h3", "h4", "h5", "h6"}

# Tags que consideramos "contenedores de bloque": si un
# elemento tiene alguno de estos como descendiente, NO es
# una hoja de contenido (para evitar duplicar texto).
TAGS_BLOQUE = {
    "h1", "h2", "h3", "h4", "h5", "h6",
    "p", "li", "div", "ul", "ol",
    "table", "section", "article", "blockquote"
}

# Tags que evaluamos como posibles hojas de contenido.
TAGS_CANDIDATAS = [
    "h1", "h2", "h3", "h4", "h5", "h6",
    "p", "li", "div", "span", "a"
]


# ==========================================================
# 1. FUNCIONES AUXILIARES YA EXISTENTES (sin cambios)
# ==========================================================

def nombre_archivo_markdown(titulo):
    return normalizar_identificador(titulo) + ".md"


def extraer_texto_limpio(elemento):
    if elemento is None:
        return ""
    texto = elemento.get_text(" ", strip=True)
    return limpiar_texto(texto)


def descargar_soup(url):
    """
    Descarga una página. Devuelve (soup, es_html).
    Si el recurso no es HTML (PDF, vídeo, etc.) es_html
    será False y soup será None.
    """

    respuesta = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    respuesta.raise_for_status()

    content_type = respuesta.headers.get("Content-Type", "")

    if "html" not in content_type.lower():
        return None, False

    return (
        BeautifulSoup(respuesta.text, "html.parser"),
        True
    )


def limpiar_contenido_html(soup):
    for elemento in soup.find_all(
        ["script", "style", "noscript", "svg", "nav", "footer", "header"]
    ):
        elemento.decompose()

    return soup


# ==========================================================
# 2. NUEVA EXTRACCIÓN DE CONTENIDO (hojas + enlaces)
# ==========================================================

def es_hoja_de_contenido(tag):
    """
    Un elemento es "hoja de contenido" si es candidato a
    contener texto propio y no tiene, a su vez, otros
    bloques anidados dentro (evitamos así procesar el mismo
    texto dos veces, una en el contenedor y otra en el hijo).
    """

    if tag.name not in TAGS_CANDIDATAS:
        return False

    if tag.name == "a":
        # Un <a> siempre se evalúa individualmente.
        return True

    descendientes_bloque = tag.find_all(TAGS_BLOQUE, recursive=True)

    return len(descendientes_bloque) == 0


def texto_markdown_de_elemento(tag, url_pagina):
    """
    Convierte el contenido de un elemento "hoja" a texto,
    respetando los enlaces que contenga como
    [texto](url) en vez de perder el href.
    """

    partes = []

    for nodo in tag.children:

        nombre_nodo = getattr(nodo, "name", None)

        if nombre_nodo == "a":

            texto_enlace = extraer_texto_limpio(nodo)
            href = (nodo.get("href") or "").strip()

            if not texto_enlace:
                continue

            if href and not href.startswith(("javascript:", "#")):

                href_absoluta = urljoin(url_pagina, href)

                partes.append(f"[{texto_enlace}]({href_absoluta})")

            else:

                partes.append(texto_enlace)

        elif nombre_nodo is not None:

            texto = extraer_texto_limpio(nodo)

            if texto:
                partes.append(texto)

        else:

            texto = limpiar_texto(str(nodo))

            if texto:
                partes.append(texto)

    return limpiar_texto(" ".join(partes))


def extraer_bloques_contenido(contenedor, url_pagina):
    """
    Recorre el contenedor y devuelve una lista de líneas
    Markdown, capturando títulos, párrafos, ítems de lista
    y enlaces sueltos (nombres, teléfonos, emails, "Más
    información"...).
    """

    lineas = []
    linea_anterior = None

    for tag in contenedor.find_all(TAGS_CANDIDATAS, recursive=True):

        if not es_hoja_de_contenido(tag):
            continue

        if tag.name == "a":

            texto = extraer_texto_limpio(tag)
            href = (tag.get("href") or "").strip()

            if not texto:
                continue

            if href and not href.startswith(("javascript:", "#")):

                href_absoluta = urljoin(url_pagina, href)
                linea = f"[{texto}]({href_absoluta})"

            else:

                linea = texto

        else:

            texto = texto_markdown_de_elemento(tag, url_pagina)

            if not texto:
                continue

            if tag.name in TAGS_TITULO:

                nivel = int(tag.name[1])
                linea = ("#" * nivel) + " " + texto

            elif tag.name == "li":

                linea = f"- {texto}"

            else:

                linea = texto

        # Evitamos duplicar la misma línea si aparece
        # justo después (artefacto habitual de anidamiento).
        if linea == linea_anterior:
            continue

        lineas.append(linea)
        linea_anterior = linea

    return lineas


def contar_palabras(lineas):
    return sum(len(linea.split()) for linea in lineas)


# ==========================================================
# 2bis. FILTRADO DE PLANTILLA (menú, migas, pie, widgets)
#
# Es el mismo menú y el mismo pie en TODO upv.es (plantilla
# nueva y antigua), así que esta lista sirve para cualquier
# sección que scrapees más adelante, no solo institución.
# ==========================================================

def normalizar_para_comparar(texto):

    texto = texto.strip().lower()
    texto = texto.strip("¡¿!?: ")

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


# Textos de menú, accesos rápidos y pie de página que
# aparecen igual en (casi) cualquier página de upv.es.
TEXTOS_BOILERPLATE = {
    "accesibilidad", "mapa web", "buscar", "directorio",
    "iniciar sesion", "emergencias", "inicio upv",
    "admision", "estudios", "investigacion", "organizacion",
    "comunidad upv",
    "admision a grado", "admision a master", "admision a doctorado",
    "internacional",
    "estudios de grado", "estudios de posgrado", "oferta academica",
    "estructuras de investigacion", "iniciativas de i+d+i", "innovacion",
    "la institucion", "vida universitaria", "escuelas y facultades",
    "departamentos", "servicios universitarios",
    "estudiante", "pas, pdi y pi", "ptgas, pdi y pi", "prensa", "titulados",
    "alumni upv", "orientador",
    "como llegar", "planos", "planos 2d", "contacto",
    "habla con nosotros",
    "tienes dudas", "contacta con nosotros",
    "quieres enviar una sugerencia, queja o felicitacion",
    "no has encontrado lo que buscas",
    "dinos que opinas", "consultanos",
    "sala de prensa", "noticias de la upv", "buscar un cargo docente",
    "area de comunicacion", "transparencia", "perfil del contratante",
    "aviso legal", "politica de cookies", "politica de privacidad",
    "gestion de cookies", "descarga nuestras apps",
}

# Estos solo cuentan como corte de plantilla si aparecen
# como TÍTULO (línea que empieza por #), para no arriesgarnos
# a cortar contenido real que use esa misma palabra suelta.
TITULOS_CORTE_PLANTILLA = {
    "esto te interesa", "recursos", "instalaciones", "media",
}


def es_linea_boilerplate(linea):

    texto_plano = re.sub(r"^#+\s*", "", linea)
    texto_plano = re.sub(r"^-\s*", "", texto_plano)

    match_enlace = re.match(r"^\[([^\]]+)\]\([^)]+\)$", texto_plano.strip())
    if match_enlace:
        texto_plano = match_enlace.group(1)

    normalizado = normalizar_para_comparar(texto_plano)

    if normalizado in TEXTOS_BOILERPLATE:
        return True

    if "universitat politecnica de valencia" in normalizado and "©" in linea:
        return True

    if re.match(r"^tel\.?\s*\(?\+?34", normalizado):
        return True

    if "::" in linea:
        return True

    if linea.startswith("#") and normalizado in TITULOS_CORTE_PLANTILLA:
        # Señal de corte: todo lo que va desde aquí es pie de
        # página / widget promocional. Se gestiona en
        # recortar_desde_titulo_plantilla, aquí solo la marcamos.
        return True

    return False


def recortar_desde_primer_h1(lineas):
    """
    El contenido real de cualquier página de upv.es (plantilla
    nueva o antigua) empieza en su <h1>. Todo lo anterior es
    menú, migas de pan o barra lateral de navegación.
    """

    for indice, linea in enumerate(lineas):

        if linea.startswith("# "):
            return lineas[indice:]

    return lineas


def recortar_en_titulo_plantilla(lineas):
    """
    Corta el documento en cuanto aparece un título que marca
    el inicio de pie de página / widgets promocionales
    ("¡Esto te interesa!", "Recursos", "Instalaciones", "Media").
    """

    for indice, linea in enumerate(lineas):

        if not linea.startswith("#"):
            continue

        texto_titulo = re.sub(r"^#+\s*", "", linea)
        normalizado = normalizar_para_comparar(texto_titulo)

        if normalizado in TITULOS_CORTE_PLANTILLA:
            return lineas[:indice]

    return lineas


def deduplicar_global(lineas):
    """
    Elimina líneas exactamente repetidas en todo el documento
    (no solo consecutivas), útil para plantillas antiguas que
    duplican el pie de página completo.
    """

    resultado = []
    vistos = set()

    for linea in lineas:

        if linea in vistos:
            continue

        vistos.add(linea)
        resultado.append(linea)

    return resultado


def limpiar_lineas_finales(lineas, recortar_h1=True):
    """
    Aplica, en orden: recorte a partir del primer <h1> (salvo
    que recortar_h1=False, útil en la página padre que ya
    gestiona su propio título), corte en los títulos de pie
    de página / widgets promocionales, filtrado de líneas de
    menú/pie sueltas, y deduplicado global. "Sabías que" NO
    se filtra (contiene datos reales de la UPV); solo se
    corta todo lo que va después de "¡Esto te interesa!",
    "Recursos", "Instalaciones" o "Media".
    """

    if recortar_h1:
        lineas = recortar_desde_primer_h1(lineas)

    lineas = recortar_en_titulo_plantilla(lineas)

    lineas = [
        linea for linea in lineas
        if not es_linea_boilerplate(linea)
    ]

    lineas = deduplicar_global(lineas)

    return lineas


# ==========================================================
# 3. EXPANSIÓN DE ENLACES HIJOS (páginas con poco contenido)
# ==========================================================

def es_url_valida_para_expandir(href, url_pagina, urls_ya_usadas):

    if not href:
        return False

    href = href.strip()

    if href.startswith(("mailto:", "tel:", "javascript:", "#")):
        return False

    absoluta = urljoin(url_pagina, href).split("#")[0]

    if absoluta == url_pagina.split("#")[0]:
        return False

    if absoluta in urls_ya_usadas:
        return False

    dominio = urlparse(absoluta).netloc

    if "upv.es" not in dominio:
        return False

    return True


def obtener_enlaces_hijos(contenedor, url_pagina, maximo=MAX_ENLACES_HIJOS):

    enlaces = []
    urls_vistas = set()

    for a in contenedor.find_all("a", href=True):

        href = a["href"]

        if not es_url_valida_para_expandir(href, url_pagina, urls_vistas):
            continue

        absoluta = urljoin(url_pagina, href).split("#")[0]
        texto = extraer_texto_limpio(a)

        if not texto:
            continue

        urls_vistas.add(absoluta)
        enlaces.append((texto, absoluta))

        if len(enlaces) >= maximo:
            break

    return enlaces


def resumir_pagina_hija(url):
    """
    Descarga una página hija y devuelve un fragmento corto
    de su contenido en Markdown, o None si no se puede.
    """

    try:

        soup, es_html = descargar_soup(url)

        if not es_html:
            return None

        soup = limpiar_contenido_html(soup)

        contenedor = (
            soup.find(id="smooth-wrapper")
            or soup.find("main")
            or soup.body
        )

        if contenedor is None:
            return None

        lineas = extraer_bloques_contenido(contenedor, url)
        lineas = limpiar_lineas_finales(lineas)

        fragmento = "\n\n".join(lineas)

        if len(fragmento) > MAX_CARACTERES_FRAGMENTO_HIJO:
            fragmento = fragmento[:MAX_CARACTERES_FRAGMENTO_HIJO].rstrip() + "…"

        return fragmento or None

    except Exception:

        return None


# ==========================================================
# 4. METADATOS YAML
# ==========================================================

def generar_yaml_metadatos(seccion_id, recurso, tipo_documento="recurso"):

    campos = [
        ("fuente", FUENTE),
        ("categoria", CATEGORIA),
        ("nivel", NIVEL),
        ("tipo_documento", tipo_documento),
        ("tipo_recurso", "informacion"),
        ("padre", PADRE_SLUG),
        ("seccion", seccion_id),
        ("url", recurso.get("url", "")),
    ]

    lineas = [f"{clave}: {valor}" for clave, valor in campos]

    return "---\n" + "\n\n".join(lineas) + "\n---\n"


# ==========================================================
# 5. GENERAR MARKDOWN DE CADA RECURSO
# ==========================================================

def generar_markdown_recurso(recurso, carpeta, seccion_id):

    titulo = recurso.get("titulo", "")
    url = recurso.get("url", "")

    if not titulo or not url:
        return False

    print()
    print(f"  Extrayendo: {titulo}")
    print(f"  URL: {url}")

    try:

        soup, es_html = descargar_soup(url)

        yaml_metadatos = generar_yaml_metadatos(seccion_id, recurso)

        # ----------------------------------------------
        # Recurso que no es HTML (PDF, vídeo, etc.)
        # ----------------------------------------------

        if not es_html:

            markdown = (
                f"{yaml_metadatos}\n"
                f"# {titulo}\n\n"
                f"**URL:** {url}\n\n"
                f"_Este recurso no es una página HTML estándar "
                f"(por ejemplo, un PDF o un vídeo). "
                f"Consulta el contenido directamente en la URL indicada._\n"
            )

            nombre_archivo = nombre_archivo_markdown(titulo)
            ruta_archivo = os.path.join(carpeta, nombre_archivo)

            with open(ruta_archivo, "w", encoding="utf-8") as archivo:
                archivo.write(markdown)

            print(f"  OK (no HTML): {ruta_archivo}")

            return True

        soup = limpiar_contenido_html(soup)

        contenido = (
            soup.find(id="smooth-wrapper")
            or soup.find("main")
            or soup.body
        )

        if contenido is None:
            print("  AVISO: no se ha encontrado contenido.")
            return False

        lineas_contenido = extraer_bloques_contenido(contenido, url)
        lineas_contenido = limpiar_lineas_finales(lineas_contenido)

        if not lineas_contenido:
            print("  AVISO: contenido vacío.")
            return False

        # ----------------------------------------------
        # Si hay poco contenido, ampliamos con enlaces
        # hijos encontrados en el propio contenedor.
        # ----------------------------------------------

        if contar_palabras(lineas_contenido) < UMBRAL_PALABRAS_POCO_CONTENIDO:

            enlaces_hijos = obtener_enlaces_hijos(contenido, url)

            if enlaces_hijos:

                lineas_contenido.append("## Información relacionada")

                for texto_enlace, url_hija in enlaces_hijos:

                    time.sleep(0.5)

                    fragmento = resumir_pagina_hija(url_hija)

                    lineas_contenido.append(f"### {texto_enlace}")
                    lineas_contenido.append(f"**URL:** {url_hija}")

                    if fragmento:
                        lineas_contenido.append(fragmento)

        markdown_contenido = "\n\n".join(lineas_contenido)

        markdown = (
            f"{yaml_metadatos}\n"
            f"# {titulo}\n\n"
            f"**URL:** {url}\n\n"
            f"{markdown_contenido}\n"
        )

        nombre_archivo = nombre_archivo_markdown(titulo)
        ruta_archivo = os.path.join(carpeta, nombre_archivo)

        with open(ruta_archivo, "w", encoding="utf-8") as archivo:
            archivo.write(markdown)

        print(f"  OK: {ruta_archivo}")

        return True

    except Exception as error:

        print(f"  ERROR: {error}")
        return False


# ==========================================================
# 6. MAPEO DE SECCIONES (id del JSON) A CARPETAS
# ==========================================================

CARPETAS_MARKDOWN = {

    "organos_de_gobierno": CARPETA_ORGANOS_GOBIERNO,
    "publicaciones_oficiales": CARPETA_PUBLICACIONES_OFICIALES,
    "la_upv_al_detalle": CARPETA_UPV_AL_DETALLE,
    "estrategia_upv_sirve": CARPETA_ESTRATEGIA_UPV_SIRVE,
    "sindicatura": CARPETA_SINDICATURA
}


# ==========================================================
# 7. PROCESAR TODAS LAS SECCIONES DEL JSON
# ==========================================================

def generar_markdowns_recursos():

    print()
    print("=" * 70)
    print("GENERANDO MARKDOWNS DE LOS RECURSOS")
    print("=" * 70)

    total = 0
    correctos = 0
    errores = 0

    for seccion in json_institucion.get("secciones", []):

        seccion_id = seccion.get("id", "")

        carpeta = CARPETAS_MARKDOWN.get(seccion_id)

        if carpeta is None:

            print()
            print(f"AVISO: no existe carpeta para id '{seccion_id}'")
            continue

        print()
        print(f"[{seccion['titulo']}]")

        for recurso in seccion.get("elementos", []):

            total += 1

            if generar_markdown_recurso(recurso, carpeta, seccion_id):
                correctos += 1
            else:
                errores += 1

    return total, correctos, errores


# ==========================================================
# 8. GENERAR MARKDOWN DE LA PÁGINA PADRE
#    (se mantiene tu lógica, solo se le añade el YAML
#    homogéneo al principio)
# ==========================================================

def generar_markdown_padre():

    print()
    print("=" * 70)
    print("GENERANDO MARKDOWN DE LA PÁGINA PADRE")
    print("=" * 70)

    soup, es_html = descargar_soup(URL_RAIZ)
    soup = limpiar_contenido_html(soup)

    contenedor = soup.find(id="smooth-wrapper")

    if contenedor is None:
        raise Exception("No se ha encontrado #smooth-wrapper.")

    lineas_contenido = extraer_bloques_contenido(contenedor, URL_RAIZ)
    lineas_contenido = limpiar_lineas_finales(lineas_contenido, recortar_h1=False)

    resultado = [limpiar_texto(linea) for linea in lineas_contenido if limpiar_texto(linea)]

    yaml_metadatos = generar_yaml_metadatos(
        seccion_id="",
        recurso={"url": URL_RAIZ},
        tipo_documento="padre"
    )

    markdown = (
        f"{yaml_metadatos}\n"
        "# La institución\n\n"
        + "\n\n".join(resultado)
        + "\n"
    )

    with open(RUTA_MARKDOWN_PADRE, "w", encoding="utf-8") as archivo:
        archivo.write(markdown)

    print()
    print("OK: Markdown padre generado:")
    print(RUTA_MARKDOWN_PADRE)

    return markdown


# ==========================================================
# 9. EJECUCIÓN
# ==========================================================

markdown_padre = generar_markdown_padre()

(
    total_recursos,
    recursos_correctos,
    recursos_error
) = generar_markdowns_recursos()


# ==========================================================
# 10. RESUMEN FINAL
# ==========================================================

print()
print("=" * 70)
print("GENERACIÓN DE MARKDOWN FINALIZADA")
print("=" * 70)

print()
print("Markdown página padre:")
print(RUTA_MARKDOWN_PADRE)

print()
print("Recursos procesados:", total_recursos)
print("Markdowns generados:", recursos_correctos)
print("Errores:", recursos_error)

print()
print("Carpeta principal:")
print(CARPETA_INSTITUCION)

print()
print("=" * 70)


GENERANDO MARKDOWN DE LA PÁGINA PADRE

OK: Markdown padre generado:
/content/drive/MyDrive/TFG Teleco/INSTITUCION/institucion.md

GENERANDO MARKDOWNS DE LOS RECURSOS

[Órganos de gobierno]

  Extrayendo: Unipersonales y colegiados
  URL: https://www.upv.es/organizacion/la-institucion/organos-gobierno-upv-es.html
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/unipersonales_y_colegiados.md

  Extrayendo: Rector
  URL: https://rector.upv.es/es
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/rector.md

  Extrayendo: Equipo rectoral
  URL: https://www.upv.es/organizacion/la-institucion/equipo-rectoral-es.html
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/equipo_rectoral.md

  Extrayendo: Vicerrectorados y sus áreas
  URL: https://www.upv.es/organizacion/la-institucion/vicerrectorados-y-areas-es.html
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/vicerrectorados_y_sus_areas.md

  Extrayendo: Secretaria G

In [ ]:
# ==========================================================
# BLOQUE EXTRA. GENERAR ÚNICAMENTE ÓRGANOS DE GOBIERNO
#
# No modifica los Markdown de las demás secciones.
# ==========================================================

print()
print("=" * 70)
print("GENERANDO MARKDOWNS - ÓRGANOS DE GOBIERNO")
print("=" * 70)


# ==========================================================
# 1. OBTENER LA SECCIÓN DEL JSON
# ==========================================================

seccion_organos = None

for seccion in json_institucion.get(
    "secciones",
    []
):

    if seccion.get("tipo") == "organos_gobierno":

        seccion_organos = seccion
        break


if seccion_organos is None:

    raise Exception(
        "No se ha encontrado la sección "
        "'organos_gobierno' en el JSON."
    )


# ==========================================================
# 2. CARPETA DE DESTINO
# ==========================================================

carpeta_organos = CARPETA_ORGANOS_GOBIERNO


print()
print("Carpeta de destino:")
print(carpeta_organos)


if not os.path.isdir(carpeta_organos):

    raise Exception(
        "La carpeta de destino no existe:\n"
        + carpeta_organos
    )


# ==========================================================
# 3. PROCESAR RECURSOS
# ==========================================================

recursos = seccion_organos.get(
    "elementos",
    []
)

total = len(recursos)
correctos = 0
errores = 0


print()
print(
    "Recursos encontrados:",
    total
)


for recurso in recursos:

    if generar_markdown_recurso(
        recurso,
        carpeta_organos
    ):

        correctos += 1

    else:

        errores += 1


# ==========================================================
# 4. RESUMEN
# ==========================================================

print()
print("=" * 70)
print("ÓRGANOS DE GOBIERNO - FINALIZADO")
print("=" * 70)

print()
print(
    "Recursos encontrados:",
    total
)

print(
    "Markdowns generados:",
    correctos
)

print(
    "Errores:",
    errores
)

print()
print(
    "Carpeta:",
    carpeta_organos
)

print()
print("=" * 70)



GENERANDO MARKDOWNS - ÓRGANOS DE GOBIERNO

Carpeta de destino:
/content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno

Recursos encontrados: 7

  Extrayendo: Unipersonales y colegiados
  URL: https://www.upv.es/organizacion/la-institucion/organos-gobierno-upv-es.html
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/unipersonales_y_colegiados.md

  Extrayendo: Rector
  URL: https://rector.upv.es/es
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/rector.md

  Extrayendo: Equipo rectoral
  URL: https://www.upv.es/organizacion/la-institucion/equipo-rectoral-es.html
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/equipo_rectoral.md

  Extrayendo: Vicerrectorados y sus áreas
  URL: https://www.upv.es/organizacion/la-institucion/vicerrectorados-y-areas-es.html
  OK: /content/drive/MyDrive/TFG Teleco/INSTITUCION/organos_gobierno/vicerrectorados_y_sus_areas.md

  Extrayendo: Secretaria General
  URL: https://www.upv.es/entid

In [ ]:
# ==========================================================
# BLOQUE 11. CORRECCIÓN Y MEJORA DEFINITIVA
#
# Este bloque modifica EXCLUSIVAMENTE:
#
#   INSTITUCION/organos_gobierno/
#
# Actuaciones:
#
# 1. Corrige los metadatos YAML de los 7 recursos.
# 2. Regenera completamente equipo_rectoral.md:
#       - rector
#       - nombre
#       - cargo
#       - teléfono
#       - correo
#       - vicerrectorados
#       - personas
# 3. Mejora gerencia.md:
#       - conserva el contenido existente
#       - resuelve todos los enlaces "Más info"
#       - añade sus URLs
#       - intenta acceder a cada recurso
#       - añade una descripción cuando puede obtenerla
#
# NO MODIFICA ninguna otra carpeta.
# ==========================================================


import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


print()
print("=" * 70)
print("BLOQUE 11. CORRECCIÓN Y MEJORA DEFINITIVA")
print("=" * 70)


# ==========================================================
# 1. CONFIGURACIÓN
# ==========================================================

CARPETA_ORGANOS = os.path.join(
    ruta_programa,
    "INSTITUCION",
    "organos_gobierno"
)


URL_EQUIPO_RECTORAL = (
    "https://www.upv.es/organizacion/la-institucion/"
    "equipo-rectoral-es.html"
)


URL_GERENCIA = (
    "https://www.upv.es/entidades/ger/"
)


PADRE = "la_institucion"
SECCION = "organos_de_gobierno"


# ==========================================================
# 2. FUNCIONES AUXILIARES
# ==========================================================


def limpiar_texto_11(texto):
    """
    Limpia espacios y saltos de línea.
    """

    if texto is None:
        return ""

    texto = str(texto)

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


def limpiar_url_11(url):
    """
    Normaliza URLs que puedan venir representadas como
    Markdown:

        [https://ejemplo.es](https://ejemplo.es)

    y devuelve únicamente:

        https://ejemplo.es
    """

    if not url:
        return ""

    url = str(url).strip()

    # Formato Markdown [texto](url)
    coincidencia = re.match(
        r"^\[.*?\]\((.*?)\)$",
        url
    )

    if coincidencia:
        url = coincidencia.group(1)

    return url.strip()


def descargar_soup_11(url):
    """
    Descarga una URL y devuelve BeautifulSoup.

    No utiliza raise_for_status(), de forma que un recurso
    que devuelva 400/403/404 no impide conservar su URL.
    """

    respuesta = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    respuesta.encoding = (
        respuesta.apparent_encoding
        or respuesta.encoding
    )

    soup = BeautifulSoup(
        respuesta.text,
        "html.parser"
    )

    return respuesta, soup


def extraer_contenedor_principal_11(soup):
    """
    Intenta localizar el contenido principal de una página.
    """

    # Prioridad a estructuras habituales de UPV
    candidatos = [
        soup.find(
            id="smooth-wrapper"
        ),
        soup.find(
            "main"
        ),
        soup.find(
            class_=re.compile(
                r"content|contenido|main",
                re.I
            )
        ),
        soup.body
    ]

    for candidato in candidatos:

        if candidato is not None:
            return candidato

    return soup


def extraer_texto_util_11(elemento):
    """
    Extrae párrafos y encabezados relevantes.
    """

    if elemento is None:
        return ""

    lineas = []

    for hijo in elemento.find_all(
        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p"
        ]
    ):

        texto = limpiar_texto_11(
            hijo.get_text(
                " ",
                strip=True
            )
        )

        if not texto:
            continue

        # Evitar textos de navegación y similares
        if texto.lower() in {
            "más info",
            "más información",
            "info",
            "acceder",
            "ver más"
        }:
            continue

        lineas.append(
            texto
        )

    # Eliminar duplicados manteniendo orden
    resultado = []
    vistos = set()

    for linea in lineas:

        if linea in vistos:
            continue

        vistos.add(linea)

        resultado.append(
            linea
        )

    return "\n\n".join(
        resultado
    )


def obtener_nombre_archivo_11(titulo):
    """
    Genera nombre estable para Markdown.
    """

    if "normalizar_identificador" in globals():

        return (
            normalizar_identificador(titulo)
            + ".md"
        )

    texto = limpiar_texto_11(
        titulo
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9áéíóúüñ]+",
        "_",
        texto
    )

    return texto.strip("_") + ".md"


# ==========================================================
# 3. METADATOS YAML
# ==========================================================


def construir_yaml_11(url):
    """
    Construye SIEMPRE los mismos ocho campos de metadatos.
    """

    url = limpiar_url_11(url)

    return (
        "---\n"
        "fuente: UPV\n\n"
        "categoria: institucion\n\n"
        "nivel: institucional\n\n"
        "tipo_documento: recurso\n\n"
        "tipo_recurso: informacion\n\n"
        f"padre: {PADRE}\n\n"
        f"seccion: {SECCION}\n\n"
        f"url: {url}\n"
        "---\n\n"
    )


def escribir_markdown_con_yaml_11(
    ruta,
    contenido,
    url
):
    """
    Elimina cualquier YAML anterior y escribe uno nuevo.
    """

    contenido = contenido.lstrip()

    # Eliminar front matter YAML previo
    if contenido.startswith("---"):

        partes = contenido.split(
            "---",
            2
        )

        if len(partes) == 3:
            contenido = partes[2].lstrip()

    yaml = construir_yaml_11(
        url
    )

    with open(
        ruta,
        "w",
        encoding="utf-8"
    ) as archivo:

        archivo.write(
            yaml
        )

        archivo.write(
            contenido.rstrip()
            + "\n"
        )


# ==========================================================
# 4. CORREGIR METADATOS DE LOS SIETE RECURSOS
# ==========================================================


def corregir_metadatos_11():

    print()
    print("=" * 70)
    print("CORRIGIENDO METADATOS YAML")
    print("=" * 70)

    archivos = [

        (
            "unipersonales_y_colegiados.md",
            "https://www.upv.es/organizacion/"
            "la-institucion/organos-gobierno-upv-es.html"
        ),

        (
            "rector.md",
            "https://rector.upv.es/es"
        ),

        (
            "equipo_rectoral.md",
            URL_EQUIPO_RECTORAL
        ),

        (
            "vicerrectorados_y_sus_areas.md",
            "https://www.upv.es/organizacion/"
            "la-institucion/vicerrectorados-y-areas-es.html"
        ),

        (
            "secretaria_general.md",
            "https://www.upv.es/entidades/SG/index-es.html"
        ),

        (
            "gerencia.md",
            URL_GERENCIA
        ),

        (
            "consejo_social.md",
            "https://www.upv.es/entidades/CS/index-es.html"
        )
    ]

    for nombre, url in archivos:

        ruta = os.path.join(
            CARPETA_ORGANOS,
            nombre
        )

        if not os.path.exists(ruta):

            print(
                f"AVISO: no existe {nombre}"
            )

            continue

        with open(
            ruta,
            "r",
            encoding="utf-8"
        ) as archivo:

            contenido = archivo.read()

        escribir_markdown_con_yaml_11(
            ruta,
            contenido,
            url
        )

        print(
            f"OK: {nombre}"
        )


# ==========================================================
# 5. EXTRAER EQUIPO RECTORAL
# ==========================================================


def extraer_equipo_rectoral_11():

    print()
    print("=" * 70)
    print("EXTRAYENDO EQUIPO RECTORAL")
    print("=" * 70)

    respuesta, soup = descargar_soup_11(
        URL_EQUIPO_RECTORAL
    )

    if respuesta.status_code >= 400:

        raise Exception(
            f"HTTP {respuesta.status_code}"
        )

    personas = []

    # ------------------------------------------------------
    # La estructura REAL de la página es:
    #
    # div.governing-section
    #     div.governing-card
    #         h3
    #         div.profile-card
    #             a.profile-card--content--title
    #             p.profile-card--content--post
    #             ...
    #
    # Por tanto NO buscamos clases genéricas de persona.
    # ------------------------------------------------------

    tarjetas = soup.select(
        "div.governing-section div.profile-card"
    )

    print(
        "Tarjetas profile-card encontradas:",
        len(tarjetas)
    )

    for tarjeta in tarjetas:

        # --------------------------------------------------
        # Cargo / vicerrectorado
        # --------------------------------------------------

        elemento_titulo = tarjeta.select_one(
            ".profile-card--content--title"
        )

        titulo = ""

        if elemento_titulo is not None:

            titulo = limpiar_texto_11(
                elemento_titulo.get_text(
                    " ",
                    strip=True
                )
            )

        # --------------------------------------------------
        # Cargo de la persona
        # --------------------------------------------------

        elemento_puesto = tarjeta.select_one(
            ".profile-card--content--post"
        )

        puesto = ""

        if elemento_puesto is not None:

            puesto = limpiar_texto_11(
                elemento_puesto.get_text(
                    " ",
                    strip=True
                )
            )

        # --------------------------------------------------
        # Contenido completo de la tarjeta
        # --------------------------------------------------

        texto_tarjeta = limpiar_texto_11(
            tarjeta.get_text(
                " ",
                strip=True
            )
        )

        # --------------------------------------------------
        # Buscar correo
        # --------------------------------------------------

        email = ""

        enlace_email = tarjeta.find(
            "a",
            href=re.compile(
                r"^mailto:",
                re.I
            )
        )

        if enlace_email is not None:

            email = limpiar_texto_11(
                enlace_email.get_text(
                    " ",
                    strip=True
                )
            )

            if not email:

                email = (
                    enlace_email.get(
                        "href",
                        ""
                    )
                    .replace(
                        "mailto:",
                        "",
                        1
                    )
                )

        # --------------------------------------------------
        # Buscar teléfono
        # --------------------------------------------------

        telefono = ""

        coincidencia_telefono = re.search(
            r"\+34\s*"
            r"\d{3}\s*"
            r"\d{3}\s*"
            r"\d{3}"
            r"(?:\s*\(ext\.?:?\s*\d+\))?",
            texto_tarjeta
        )

        if coincidencia_telefono:

            telefono = limpiar_texto_11(
                coincidencia_telefono.group(0)
            )

        # --------------------------------------------------
        # Extraer nombre
        #
        # El nombre NO tiene una clase propia.
        #
        # Lo obtenemos eliminando del texto de la tarjeta:
        # - título
        # - puesto
        # - teléfono
        # - correo
        # --------------------------------------------------

        nombre = texto_tarjeta

        partes_a_eliminar = [
            titulo,
            puesto,
            telefono,
            email
        ]

        for parte in partes_a_eliminar:

            if parte:

                nombre = nombre.replace(
                    parte,
                    " "
                )

        nombre = limpiar_texto_11(
            nombre
        )

        # --------------------------------------------------
        # Eliminar textos accesorios
        # --------------------------------------------------

        nombre = re.sub(
            r"Fotografía.*$",
            "",
            nombre,
            flags=re.I
        )

        nombre = limpiar_texto_11(
            nombre
        )

        # --------------------------------------------------
        # Si por la estructura concreta del rector queda
        # algún "Rector", eliminarlo si no es realmente
        # el nombre.
        # --------------------------------------------------

        if nombre.lower() in {
            "",
            "rector",
            "vicerrectora",
            "vicerrector",
            "secretario general",
            "gerente"
        }:

            nombre = ""

        # --------------------------------------------------
        # Guardar solo si tenemos nombre
        # --------------------------------------------------

        if nombre:

            personas.append(
                {
                    "titulo": titulo,
                    "nombre": nombre,
                    "puesto": puesto,
                    "telefono": telefono,
                    "email": email
                }
            )

    print()
    print(
        "Personas detectadas:",
        len(personas)
    )

    for persona in personas:

        print(
            "  -",
            persona["nombre"]
        )

    if not personas:

        raise Exception(
            "No se han podido extraer personas "
            "del equipo rectoral."
        )

    return personas


# ==========================================================
# 6. GENERAR EQUIPO RECTORAL.MD
# ==========================================================


def generar_equipo_rectoral_11():

    personas = extraer_equipo_rectoral_11()

    lineas = []

    lineas.append(
        "# Equipo rectoral"
    )

    # ------------------------------------------------------
    # Agrupar por bloques según el título
    # ------------------------------------------------------

    grupo_actual = None

    for persona in personas:

        titulo = persona["titulo"]

        # ----------------------------------------------
        # Determinar encabezado de grupo
        # ----------------------------------------------

        if (
            titulo
            and titulo.lower() == "rector"
        ):

            grupo = "Rector"

        elif titulo:

            grupo = titulo

        else:

            grupo = "Equipo rectoral"

        # ----------------------------------------------
        # Nuevo grupo
        # ----------------------------------------------

        if grupo != grupo_actual:

            if grupo.lower() == "rector":

                nivel = "##"

            elif (
                "vicerrectorado"
                in grupo.lower()
            ):

                nivel = "###"

            else:

                nivel = "###"

            lineas.append(
                f"{nivel} {grupo}"
            )

            grupo_actual = grupo

        # ----------------------------------------------
        # Nombre
        # ----------------------------------------------

        lineas.append(
            f"**{persona['nombre']}**"
        )

        # ----------------------------------------------
        # Puesto
        # ----------------------------------------------

        if persona["puesto"]:

            lineas.append(
                persona["puesto"]
            )

        # ----------------------------------------------
        # Teléfono
        # ----------------------------------------------

        if persona["telefono"]:

            lineas.append(
                f"Teléfono: {persona['telefono']}"
            )

        # ----------------------------------------------
        # Correo
        # ----------------------------------------------

        if persona["email"]:

            lineas.append(
                f"Correo: {persona['email']}"
            )

    contenido = (
        "\n\n".join(
            lineas
        )
        + "\n"
    )

    ruta = os.path.join(
        CARPETA_ORGANOS,
        "equipo_rectoral.md"
    )

    escribir_markdown_con_yaml_11(
        ruta,
        contenido,
        URL_EQUIPO_RECTORAL
    )

    print()
    print(
        "OK:",
        ruta
    )


# ==========================================================
# 7. EXTRAER INFORMACIÓN DE ENLACES "MÁS INFO"
# ==========================================================


def extraer_info_enlace_11(
    url
):

    url = limpiar_url_11(
        url
    )

    try:

        respuesta, soup = descargar_soup_11(
            url
        )

        # --------------------------------------------------
        # Aunque haya HTTP >= 400, no descartamos la URL.
        # --------------------------------------------------

        if respuesta.status_code >= 400:

            return {
                "url": url,
                "texto": "",
                "estado": respuesta.status_code
            }

        contenido = (
            extraer_contenedor_principal_11(
                soup
            )
        )

        texto = extraer_texto_util_11(
            contenido
        )

        # --------------------------------------------------
        # Quitar encabezados repetidos
        # --------------------------------------------------

        lineas = texto.split(
            "\n\n"
        )

        resultado = []

        for linea in lineas:

            linea = limpiar_texto_11(
                linea
            )

            if not linea:
                continue

            if linea.lower() in {
                "más info",
                "más información",
                "información general"
            }:
                continue

            resultado.append(
                linea
            )

        # Nos quedamos con un bloque razonable de texto
        # para no duplicar una página completa dentro de
        # gerencia.md.
        resultado = resultado[:8]

        texto = "\n\n".join(
            resultado
        )

        return {
            "url": url,
            "texto": texto,
            "estado": respuesta.status_code
        }

    except Exception as error:

        return {
            "url": url,
            "texto": "",
            "estado": str(error)
        }


# ==========================================================
# 8. MEJORAR GERENCIA
# ==========================================================


def mejorar_gerencia_11():

    print()
    print("=" * 70)
    print("MEJORANDO GERENCIA")
    print("=" * 70)

    respuesta, soup = descargar_soup_11(
        URL_GERENCIA
    )

    if respuesta.status_code >= 400:

        raise Exception(
            f"HTTP {respuesta.status_code}"
        )

    # ------------------------------------------------------
    # Localizar todos los enlaces "Más info"
    # ------------------------------------------------------

    enlaces = []

    for enlace in soup.find_all(
        "a"
    ):

        texto = limpiar_texto_11(
            enlace.get_text(
                " ",
                strip=True
            )
        )

        if texto.lower() not in {
            "más info",
            "más información"
        }:
            continue

        href = enlace.get(
            "href",
            ""
        )

        if not href:
            continue

        url = urljoin(
            URL_GERENCIA,
            href
        )

        # Evitar duplicados
        if any(
            item["url"] == url
            for item in enlaces
        ):
            continue

        # --------------------------------------------------
        # Buscar el título de la sección que contiene
        # el enlace.
        # --------------------------------------------------

        contenedor = enlace

        titulo = ""

        # Subir por el DOM hasta encontrar un bloque que
        # contenga un encabezado.
        for _ in range(5):

            if contenedor is None:
                break

            encabezados = contenedor.find_all(
                [
                    "h2",
                    "h3",
                    "h4"
                ]
            )

            if encabezados:

                titulo = limpiar_texto_11(
                    encabezados[-1].get_text(
                        " ",
                        strip=True
                    )
                )

                break

            contenedor = contenedor.parent

        if not titulo:

            titulo = "Información adicional"

        enlaces.append(
            {
                "titulo": titulo,
                "url": url
            }
        )

    print()
    print(
        "Enlaces 'Más info' encontrados:",
        len(enlaces)
    )

    # ------------------------------------------------------
    # Obtener información de cada enlace
    # ------------------------------------------------------

    informacion = []

    for item in enlaces:

        print(
            "  Consultando:",
            item["titulo"]
        )

        datos = extraer_info_enlace_11(
            item["url"]
        )

        datos["titulo"] = item["titulo"]

        informacion.append(
            datos
        )

    # ------------------------------------------------------
    # Ahora reconstruimos el Markdown a partir de la
    # página de Gerencia.
    # ------------------------------------------------------

    lineas = []

    lineas.append(
        "# Gerencia"
    )

    # ------------------------------------------------------
    # Extraer contenido principal
    # ------------------------------------------------------

    contenido = (
        extraer_contenedor_principal_11(
            soup
        )
    )

    # ------------------------------------------------------
    # Recorrer encabezados y párrafos
    # ------------------------------------------------------

    for elemento in contenido.find_all(
        [
            "h2",
            "h3",
            "h4",
            "p"
        ]
    ):

        texto = limpiar_texto_11(
            elemento.get_text(
                " ",
                strip=True
            )
        )

        if not texto:
            continue

        # No incluir "Más info" como texto suelto
        if texto.lower() in {
            "más info",
            "más información"
        }:
            continue

        # ----------------------------------------------
        # Detectar encabezado
        # ----------------------------------------------

        if elemento.name == "h2":

            lineas.append(
                f"## {texto}"
            )

        elif elemento.name == "h3":

            lineas.append(
                f"### {texto}"
            )

        elif elemento.name == "h4":

            lineas.append(
                f"#### {texto}"
            )

        else:

            lineas.append(
                texto
            )

    # ------------------------------------------------------
    # Eliminar duplicados conservando orden
    # ------------------------------------------------------

    resultado = []
    vistos = set()

    for linea in lineas:

        linea = limpiar_texto_11(
            linea
        )

        if not linea:
            continue

        if linea in vistos:
            continue

        vistos.add(
            linea
        )

        resultado.append(
            linea
        )

    # ------------------------------------------------------
    # Añadir información de los enlaces "Más info"
    #
    # Se añade al final un bloque específico con:
    #
    # ### Servicio...
    #
    # descripción
    #
    # [Más información](URL)
    # ------------------------------------------------------

    if informacion:

        resultado.append(
            "## Información adicional"
        )

        for item in informacion:

            titulo = item["titulo"]
            url = item["url"]
            texto = item["texto"]

            resultado.append(
                f"### {titulo}"
            )

            if texto:

                resultado.append(
                    texto
                )

            resultado.append(
                f"[Más información]({url})"
            )

    markdown = (
        "\n\n".join(
            resultado
        )
        + "\n"
    )

    ruta = os.path.join(
        CARPETA_ORGANOS,
        "gerencia.md"
    )

    escribir_markdown_con_yaml_11(
        ruta,
        markdown,
        URL_GERENCIA
    )

    print()
    print(
        "OK:",
        ruta
    )


# ==========================================================
# 9. EJECUCIÓN
# ==========================================================


try:

    # ------------------------------------------------------
    # Primero corregimos los metadatos de TODOS los
    # recursos.
    # ------------------------------------------------------

    corregir_metadatos_11()

    # ------------------------------------------------------
    # Después regeneramos equipo rectoral.
    # ------------------------------------------------------

    print()
    print("=" * 70)
    print("REGENERANDO EQUIPO RECTORAL")
    print("=" * 70)

    generar_equipo_rectoral_11()

    # ------------------------------------------------------
    # Finalmente regeneramos Gerencia.
    # ------------------------------------------------------

    print()
    print("=" * 70)
    print("REGENERANDO GERENCIA")
    print("=" * 70)

    mejorar_gerencia_11()

except Exception as error:

    print()
    print(
        "ERROR:",
        error
    )


# ==========================================================
# 10. COMPROBACIÓN FINAL
# ==========================================================


print()
print("=" * 70)
print("BLOQUE 11 FINALIZADO")
print("=" * 70)

print()
print(
    "Carpeta modificada:"
)

print(
    CARPETA_ORGANOS
)

print()
print(
    "Archivos modificados:"
)

print(
    "  - unipersonales_y_colegiados.md"
)

print(
    "  - rector.md"
)

print(
    "  - equipo_rectoral.md"
)

print(
    "  - vicerrectorados_y_sus_areas.md"
)

print(
    "  - secretaria_general.md"
)

print(
    "  - gerencia.md"
)

print(
    "  - consejo_social.md"
)

print()
print(
    "No se han modificado las demás carpetas."
)

print()
print("=" * 70)


BLOQUE 11. CORRECCIÓN Y MEJORA DEFINITIVA

CORRIGIENDO METADATOS YAML
OK: unipersonales_y_colegiados.md
OK: rector.md
OK: equipo_rectoral.md
OK: vicerrectorados_y_sus_areas.md
OK: secretaria_general.md
OK: gerencia.md
OK: consejo_social.md

REGENERANDO EQUIPO RECTORAL

EXTRAYENDO EQUIPO RECTORAL
Tarjetas profile-card encontradas: 16

Personas detectadas: 16
  - José Esteban Capilla Romá + 34 963877100 (ext.: 71000) rector@upv.es
  - María Salomé Cuesta Valera vacts@upv.es
  - Juan Carlos Cano Escribá vpoa@upv.es
  - María Belén Picó Sirvent vinv@upv.es
  - José Pedro García Sabater vpec@upv.es
  - Débora Domingo Calabuig vcampus@upv.es
  - Eugenio Pellicer Armiñana vinfraestructuras@upv.es
  - Salvador Coll Arnau vit@upv.es
  - J. Alberto Conejero Casares vee@upv.es
  - José Francisco Monserrat Del Río vic@upv.es
  - Silvia Mª Terrasa Barrena vtdigital@upv.es
  - Elena de la Poza Plaza vefpl@upv.es
  - Santiago Guillem Picó vconvivencia@upv.es
  - Josep Antoni Claver Campillo secret@u